# 蒙特卡洛基础方法（MC Basic）

## 一、导入依赖库

In [1]:
import numpy as np       # 导入 NumPy 库，用于数值计算和矩阵运算，版本要求 >=1.18
import random            # 导入 Python 标准库 random，用于随机数生成
import importlib.util    # 导入 importlib.util，用于按文件路径动态加载模块

# 文件名 "02.1.ModelFree_Env_GridWorldV2.py" 以数字开头且含点号，不符合 Python 标识符规则，无法直接 import
# spec_from_file_location：根据给定模块别名和 .py 文件路径创建模块规格，返回 ModuleSpec 对象
_spec = importlib.util.spec_from_file_location("GridWorld_v2", "02.1.ModelFree_Env_GridWorldV2.py")
# module_from_spec：根据模块规格创建模块对象，此时模块代码尚未执行，返回 module 对象
GridWorld_v2 = importlib.util.module_from_spec(_spec)
# exec_module：执行模块代码完成初始化，之后可通过 GridWorld_v2.GridWorld_v2(...) 正常使用类
_spec.loader.exec_module(GridWorld_v2)

In [2]:
np.eye(5)  # 生成 5×5 单位矩阵，np.ndarray，shape=(5,5)，对角线为 1 其余为 0；演示 one-hot 编码的矩阵基础

array([[1., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 1.]])

## 二、初始化网格世界与策略

In [3]:
gamma = 0.9   # 折扣因子 γ，float，控制未来奖励的衰减程度，越接近 0 越短视，越接近 1 越重视长期回报

rows = 5      # 网格世界的行数，int，需与 desc 描述字符串的行数保持一致
columns = 5   # 网格世界的列数，int，需与 desc 描述字符串每行字符数保持一致

# 使用描述字符串初始化网格世界：'.' 表示普通格，'#' 表示禁止区（得分 -10），'T' 表示目标格（得分 1）
gridworld = GridWorld_v2.GridWorld_v2(
    forbiddenAreaScore=-10,  # 禁止区域的即时奖励，float，负值表示惩罚
    score=1,                 # 目标区域的即时奖励，float
    desc=[".....", ".##..", "..#..", ".#T#.", ".#..."]  # 网格布局描述，list[str]，共 5 行 5 列
)
gridworld.show()  # 以 emoji 可视化打印网格世界布局，无返回值

trajectorySteps = 100  # 每次采样轨迹的最大步数，int

value = np.zeros(rows * columns)        # 初始化状态价值函数 V(s)，np.ndarray，shape=(25,)，全零
qtable = np.zeros((rows * columns, 5))  # 初始化动作价值函数 Q(s,a)，np.ndarray，shape=(25, 5)，全零

# 随机初始化确定性策略，利用 NumPy 花式索引（Fancy Indexing）一步完成整数索引→one-hot 转换：
# np.random.randint(0,5,size=(rows*columns))：生成 25 个随机动作索引，shape=(25,)，每个值∈[0,4]，int
# np.eye(5)[整数数组]：花式索引——对数组中每个整数 i，取 np.eye(5) 第 i 行（动作 i 的 one-hot 向量）
#   并沿第 0 轴堆叠，等价于 np.stack([np.eye(5)[i] for i in idx])，但由 C 实现、效率更高
# 结果 policy：np.ndarray，shape=(25, 5)，每行恰有一个 1（对应该状态随机选定的动作），其余为 0
policy = np.eye(5)[np.random.randint(0, 5, size=(rows * columns))]
print(policy)  # 打印初始随机策略矩阵，shape=(25, 5)，每行恰有一个 1，其余为 0

gridworld.showPolicy(policy)  # 以 emoji 可视化当前策略，显示每个格子的推荐动作

⬜️⬜️⬜️⬜️⬜️
⬜️🚫🚫⬜️⬜️
⬜️⬜️🚫⬜️⬜️
⬜️🚫✅🚫⬜️
⬜️🚫⬜️⬜️⬜️
[[1. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]
 [1. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 1.]
 [1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1.]
 [1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 1.]
 [1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 1. 0. 0. 0.]]
⬆️⬅️🔄⬆️⬅️
🔄⏩️⏬🔄⬆️
🔄🔄⏫️🔄➡️
⬇️⏪✅⏩️⬇️
🔄⏫️➡️⬇️➡️


## 三、MC Basic 策略迭代

In [4]:
gridworld.showPolicy(policy)  # 显示当前（随机初始化）策略在网格世界中的动作分布
print("random policy")        # 打印提示：当前为随机策略，即迭代前的初始状态

qtable = np.zeros((rows * columns, 5))  # 重新初始化 Q 表，np.ndarray，shape=(25, 5)，全零
qtable_pre = qtable.copy() + 1          # 复制 Q 表并整体加 1 作为上一轮快照，保证首轮进入循环，shape=(25, 5)

# 收敛判断：当前轮与上一轮 Q 表元素差的平方和大于 0.001 时继续迭代
while np.sum((qtable_pre - qtable) ** 2) > 0.001:
    print(np.sum((qtable_pre - qtable) ** 2))  # 打印当前 Q 表变化量（平方和），越小越接近收敛
    qtable_pre = qtable.copy()  # 将当前 Q 表保存为上一轮快照，np.ndarray，shape=(25, 5)

    for i in range(rows * columns):  # 遍历所有状态，i 为状态编号，int，范围 [0, 25)
        for j in range(5):           # 遍历所有动作，j 为动作编号，int，范围 [0, 5)
            # 从状态 i 出发执行动作 j，按当前策略 policy 采样 trajectorySteps 步轨迹
            # 返回值 Trajectory：list[tuple]，长度为 trajectorySteps+1
            # 每个元组格式：(nowState, nowAction, score, nextState, nextAction)
            Trajectory = gridworld.getTrajectoryScore(
                nowState=i, action=j, policy=policy, steps=trajectorySteps
            )

            # 取轨迹末尾（第 trajectorySteps 步）的即时奖励作为折扣回报递推的初始值，float
            tmp = Trajectory[trajectorySteps][2]

            # 【逆向递推计算折扣累计回报】
            # 折扣累计回报展开式：G_0 = r_0 + γ·r_1 + γ²·r_2 + ... + γ^T·r_T
            # 若正向遍历，每步须单独计算 γ^k（幂运算），总计 O(T²) 次乘法，效率低
            # 利用递推关系 G_k = r_k + γ·G_{k+1}，从末尾 G_T 出发逆向推算：
            #   G_T = r_T
            #   G_{T-1} = r_{T-1} + γ·G_T
            #   G_{T-2} = r_{T-2} + γ·G_{T-1}  ...  G_0 = r_0 + γ·G_1
            # 每步仅需一次乘法和一次加法，总计 O(T) 次乘法，远优于正向遍历的 O(T²)
            for k in range(trajectorySteps - 1, -1, -1):
                # Trajectory[k][2]：第 k 步即时奖励 r_k，float
                # 迭代前 tmp = G_{k+1}，迭代后 tmp = G_k = r_k + γ·G_{k+1}，float
                tmp = tmp * gamma + Trajectory[k][2]  # G_k = r_k + γ·G_{k+1}

            # 将状态-动作对 (i, j) 的折扣累计回报写入 Q 表：Q(s=i, a=j) ← G
            qtable[i][j] = tmp

    # 【贪心策略改进】对每个状态选取 Q 值最大的动作，构造新的确定性策略（one-hot 编码）
    # 分两步理解：
    #   第一步 np.argmax(qtable, axis=1)：
    #     qtable shape=(25,5)，axis=1 表示在每行（每个状态）内找最大 Q 值的列索引
    #     返回 shape=(25,) 的整数数组，每个元素∈[0,4]，即该状态的最优动作编号
    #     例：状态0的Q值为[-1, -1, 9, -1, 10] → argmax=4，即动作4最优
    #   第二步 np.eye(5)[整数数组]：
    #     np.eye(5) 是 5×5 单位矩阵，第 i 行 = 动作 i 的 one-hot 向量
    #       第0行→[1,0,0,0,0]，第1行→[0,1,0,0,0]，...，第4行→[0,0,0,0,1]
    #     用整数数组批量取行：动作编号是几就取第几行，等价于逐状态构造 one-hot
    #     结果 policy：np.ndarray，shape=(25, 5)，每行恰有一个 1（最优动作位置），其余为 0
    policy = np.eye(5)[np.argmax(qtable, axis=1)]

    print(qtable[17])  # 打印状态 17 的 Q 值向量，np.ndarray，shape=(5,)，用于调试
    print(qtable[22])  # 打印状态 22 的 Q 值向量，np.ndarray，shape=(5,)，用于调试

    gridworld.showPolicy(policy)  # 可视化当前更新后的策略
    print(f'Q差异{np.sum((qtable_pre - qtable) ** 2)}')  # 打印本轮 Q 表变化量，用于监控收敛


⬆️⬅️🔄⬆️⬅️
🔄⏩️⏬🔄⬆️
🔄🔄⏫️🔄➡️
⬇️⏪✅⏩️⬇️
🔄⏫️➡️⬇️➡️
random policy
125.0
[-99.99760947 -17.28976095  -8.09976095 -10.          -8.        ]
[ -8.          -8.99976095  -9.09976095 -19.          -8.09976095]
⬇️➡️🔄⬇️⬇️
⬇️⏬⏫️⬇️⬅️
⬆️⬅️⏩️⬆️⬅️
⬆️⏫️✅⏫️🔄
⬆️⏪⬆️⬅️⬆️
Q差异157694.799509964
157694.799509964
[-10.         -10.           8.99976095 -10.           9.99976095]
[  9.99976095   8.09976095   7.99976095 -10.           8.99976095]
➡️➡️➡️➡️⬇️
⬆️⏫️⏫️⬆️⬆️
⬆️⬅️⏬⬆️⬆️
⬆️⏩️✅⏪⬆️
⬆️⏩️⬆️⬅️⬅️
Q差异129577.85112691556
129577.85112691556
[-1.00023905 -1.00023905  8.99976095 -1.00023905  9.99976095]
[ 9.99976095  8.09976095  7.99976095 -1.90023905  8.99976095]
➡️➡️➡️➡️⬇️
⬆️⏫️⏫️⬆️⬆️
⬆️⬅️⏬⬆️⬆️
⬆️⏩️✅⏪⬇️
⬆️⏩️⬆️⬅️⬅️
Q差异1808.669167035759
1808.669167035759
[-1.00023905 -1.00023905  8.99976095 -1.00023905  9.99976095]
[ 9.99976095  8.09976095  7.99976095 -1.90023905  8.99976095]
➡️➡️➡️➡️⬇️
⬆️⏫️⏫️⬆️⬆️
⬆️⬅️⏬⬆️⬇️
⬆️⏩️✅⏪⬇️
⬆️⏩️⬆️⬅️⬅️
Q差异215.2179210453017
215.2179210453017
[-1.00023905 -1.00023905  8.99976095 -1.00023905  9.99976